# 01 - 数据预处理

本 notebook 用于演示音频数据的预处理流程，包括：
1. 加载原始音频文件
2. 音频格式转换、重采样和单声道处理
3. 将处理后的音频保存到指定目录
4. （可选）演示音频分段
5. 文本记录的处理方式说明

## 1. 初始化与配置

In [2]:
import os
import sys

import numpy as np
import pandas as pd

# import glob
# import librosa
# import soundfile as sf
# from pydub import AudioSegment
# import matplotlib.pyplot as plt
# import IPython.display as ipd

# 将项目根目录添加到 sys.path 以便导入 src 中的模块
# 假设此 notebook 在 'notebooks' 文件夹下，项目根目录是上一级
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.config import RAW_AUDIO_DIR, PROCESSED_AUDIO_DIR, TRANSCRIPTS_DIR, TARGET_SAMPLE_RATE, TARGET_CHANNELS
from src.utils.audio_utils import AudioUtils
from src.asr.audio_segmenter import AudioSegmenter # 可选，用于演示分段
from src.utils.logging_utils import get_logger

logger = get_logger('DataPreprocessingNotebook')

In [3]:
# 确保目标目录存在
os.makedirs(RAW_AUDIO_DIR, exist_ok=True)
os.makedirs(PROCESSED_AUDIO_DIR, exist_ok=True)
os.makedirs(TRANSCRIPTS_DIR, exist_ok=True)

logger.info(f"原始音频目录: {os.path.abspath(RAW_AUDIO_DIR)}")
logger.info(f"处理后音频目录: {os.path.abspath(PROCESSED_AUDIO_DIR)}")
logger.info(f"文本记录目录: {os.path.abspath(TRANSCRIPTS_DIR)}")
logger.info(f"目标采样率: {TARGET_SAMPLE_RATE} Hz")
logger.info(f"目标通道数: {TARGET_CHANNELS}")

2025-06-04 18:00:20 - DataPreprocessingNotebook - INFO - 原始音频目录: D:\project\my\research\dataset\NCSSD\C-ZH
2025-06-04 18:00:20 - DataPreprocessingNotebook - INFO - 处理后音频目录: D:\project\my\research\streamllm\data\processed_audio
2025-06-04 18:00:20 - DataPreprocessingNotebook - INFO - 文本记录目录: D:\project\my\research\streamllm\data\transcripts
2025-06-04 18:00:20 - DataPreprocessingNotebook - INFO - 目标采样率: 16000 Hz
2025-06-04 18:00:20 - DataPreprocessingNotebook - INFO - 目标通道数: 1


## 1.1 从原始数据中过滤出需要的时长范围的语音
原始音频目录的文件组织形式是这样的：
目录按序号命名，每个目录里是一次对话的录音文件及对应文本的列表。录音文件与对应的文本文件名相同，后缀名不同。录音文件后缀为.wav,文本文件后缀为.lab
文件的命名规则为：序号_说话人编号_d目录。例如：目录130中，第一个对话录音文件名为：0_0_d130.wav，对应文本文件：0_0_d130.lab。代表说话人0说了第1句话。而接下来的第2句话文件名为：1_1_d130,代表说话人1对说话人0对第一句话的回复，而接下来的第3句文件名：2_0_d130，表示说话人0对上一句也就是1_1_d130的回复，依次往下进行多轮对话。

In [5]:
import glob
from pydub import AudioSegment

def filter_audio_by_duration():
    # 创建目标目录
    duration_dirs = {
        'length10': os.path.join(PROCESSED_AUDIO_DIR, 'length10'),
        'length20': os.path.join(PROCESSED_AUDIO_DIR, 'length20'),
        'length30': os.path.join(PROCESSED_AUDIO_DIR, 'length30'),
        'length30+': os.path.join(PROCESSED_AUDIO_DIR, 'length30+')
    }
    
    for dir_path in duration_dirs.values():
        os.makedirs(dir_path, exist_ok=True)
        os.makedirs(os.path.join(TRANSCRIPTS_DIR, os.path.basename(dir_path)), exist_ok=True)
    
    # 初始化计数器
    counts = {k: 0 for k in duration_dirs.keys()}
    max_samples = 10  # 每个类别最多10条记录
    
    # 遍历所有目录
    for dir_num in range(1, 1000):  # 假设目录编号从1开始
        dir_path = os.path.join(RAW_AUDIO_DIR, str(dir_num))
        if not os.path.exists(dir_path):
            continue
            
        # 获取该目录下的所有wav文件
        wav_files = glob.glob(os.path.join(dir_path, "*.wav"))
        
        for wav_file in wav_files:
            # 如果所有类别都已达到最大样本数，则退出
            if all(count >= max_samples for count in counts.values()):
                return
            
            # 获取对应的文本文件
            lab_file = wav_file.replace('.wav', '.lab')
            if not os.path.exists(lab_file):
                continue
                
            # 获取音频时长
            audio = AudioSegment.from_wav(wav_file)
            duration_seconds = len(audio) / 1000.0  # 转换为秒
            
            # 根据时长分类
            target_dir = None
            if duration_seconds <= 10:
                target_dir = 'length10'
            elif duration_seconds <= 20:
                target_dir = 'length20'
            elif duration_seconds <= 30:
                target_dir = 'length30'
            else:
                target_dir = 'length30+'
                
            # 如果该类别还未达到最大样本数
            if counts[target_dir] < max_samples:
                # 复制音频文件
                new_wav_path = os.path.join(duration_dirs[target_dir], os.path.basename(wav_file))
                audio.export(new_wav_path, format='wav')
                
                # 复制文本文件
                new_lab_path = os.path.join(TRANSCRIPTS_DIR, target_dir, os.path.basename(lab_file))
                with open(lab_file, 'r', encoding='GBK') as f_in:
                    with open(new_lab_path, 'w', encoding='GBK') as f_out:
                        f_out.write(f_in.read())
                
                counts[target_dir] += 1
                logger.info(f"已处理: {wav_file} -> {target_dir} (时长: {duration_seconds:.2f}秒)")
    
    # 打印统计信息
    logger.info("\n处理完成，统计信息：")
    for category, count in counts.items():
        logger.info(f"{category}: {count} 个文件")

# 执行过滤
filter_audio_by_duration()


2025-06-04 18:13:13 - DataPreprocessingNotebook - INFO - 已处理: D:\project\my\research\dataset\NCSSD\C-ZH\1\0_0_d1.wav -> length10 (时长: 0.86秒)
2025-06-04 18:13:13 - DataPreprocessingNotebook - INFO - 已处理: D:\project\my\research\dataset\NCSSD\C-ZH\1\1_1_d1.wav -> length10 (时长: 1.38秒)
2025-06-04 18:13:13 - DataPreprocessingNotebook - INFO - 已处理: D:\project\my\research\dataset\NCSSD\C-ZH\1\2_0_d1.wav -> length10 (时长: 1.88秒)
2025-06-04 18:13:13 - DataPreprocessingNotebook - INFO - 已处理: D:\project\my\research\dataset\NCSSD\C-ZH\1\3_1_d1.wav -> length10 (时长: 5.28秒)
2025-06-04 18:13:13 - DataPreprocessingNotebook - INFO - 已处理: D:\project\my\research\dataset\NCSSD\C-ZH\1\4_0_d1.wav -> length10 (时长: 5.26秒)
2025-06-04 18:13:13 - DataPreprocessingNotebook - INFO - 已处理: D:\project\my\research\dataset\NCSSD\C-ZH\1\5_1_d1.wav -> length10 (时长: 1.74秒)
2025-06-04 18:13:13 - DataPreprocessingNotebook - INFO - 已处理: D:\project\my\research\dataset\NCSSD\C-ZH\2\0_0_d2.wav -> length10 (时长: 2.90秒)
2025-06-04 18

## 1.2 测试程序是否能正常运行



### 1.2.1 测试asr优化程序是否运行正确

In [ ]:
sys.path.append('..')
from src.asr.faster_whisper_streamer import FasterWhisperStreamer
from src.config import ASR_MODEL_NAME, DEVICE, VAD_PARAMETERS
import soundfile as sf


# 导入日志工具
from src.utils.logging_utils import get_logger
logger = get_logger(__name__)

def test_asr_streaming():
    """测试ASR流式处理功能"""
    logger.info("测试ASR流式处理...")
    
    # 初始化ASR模型
    asr_streamer = FasterWhisperStreamer(
        model_size=ASR_MODEL_NAME,
        device=DEVICE,
        vad_parameters=VAD_PARAMETERS
    )
    
    # 获取已处理的音频文件
    processed_dir = os.path.join(PROCESSED_AUDIO_DIR, 'length10')
    if not os.path.exists(processed_dir) or len(os.listdir(processed_dir)) == 0:
        logger.warning("没有找到已处理的length10音频文件，请先执行音频处理步骤")
        return
    
    # 选择第一个音频文件进行测试
    audio_files = [f for f in os.listdir(processed_dir) if f.endswith('.wav')]
    if not audio_files:
        logger.warning("length10目录下没有找到wav文件")
        return
    
    test_audio_path = os.path.join(processed_dir, audio_files[0])
    logger.info(f"使用音频文件进行测试: {test_audio_path}")
    
    # 加载音频
    audio_data, sample_rate = sf.read(test_audio_path)
    
    # 如果是立体声转单声道
    if len(audio_data.shape) > 1 and audio_data.shape[1] > 1:
        audio_data = audio_data.mean(axis=1)
    
    # 模拟流式输入 - 每次传入500ms的数据
    chunk_size = int(sample_rate * 0.5)  # 500ms的数据量
    
    # 初始化ASR流
    asr_streamer.start_stream()
    
    text_buffer = []
    
    logger.info("开始模拟音频流...")
    
    # 逐块处理音频数据
    for i in range(0, len(audio_data), chunk_size):
        chunk = audio_data[i:i + chunk_size]
        
        # 向ASR引擎发送音频块
        asr_streamer.process_audio_chunk(chunk, sample_rate)
        
        # 获取当前的转录结果
        text = asr_streamer.get_current_transcript()
        
        if text and text not in text_buffer:
            text_buffer.append(text)
            logger.info(f"当前转录 [{i/sample_rate:.2f}s]: {text}")
    
    # 完成流式处理
    final_text = asr_streamer.finish_stream()
    logger.info(f"最终转录结果: {final_text}")
    
    return final_text

# 执行测试
test_asr_streaming()


## 2. 加载原始音频数据

首先，我们需要一些示例原始音频文件。请将您的 `.wav`, `.mp3`, 或其他格式的音频文件放入 `data/raw_audio/` 目录下。

为了演示，我们将创建一个虚拟的原始音频文件（如果该目录为空）。在实际使用中，您应该使用自己的数据。

In [ ]:
def create_dummy_audio_if_not_exists(filename="dummy_audio.wav", duration_s=5, sr=44100, channels=2):
    dummy_file_path = os.path.join(RAW_AUDIO_DIR, filename)
    if not os.path.exists(dummy_file_path):
        logger.info(f"创建虚拟音频文件: {dummy_file_path}")
        # 创建一个简单的正弦波作为虚拟音频
        t = np.linspace(0, duration_s, int(sr * duration_s), endpoint=False)
        dummy_audio_data = 0.5 * np.sin(2 * np.pi * 440 * t) # 440 Hz 音调
        if channels == 2:
            dummy_audio_data = np.stack([dummy_audio_data, dummy_audio_data], axis=-1) # 立体声
        sf.write(dummy_file_path, dummy_audio_data, sr)
        logger.info(f"虚拟音频文件已创建: {dummy_file_path}")
    else:
        logger.info(f"音频文件 {dummy_file_path} 已存在。")
    return dummy_file_path

# 创建或获取一个虚拟音频文件用于演示
example_raw_audio_file = create_dummy_audio_if_not_exists()

# 列出原始音频目录中的所有音频文件 (示例，支持多种格式)
raw_audio_files = []
for ext in ("*.wav", "*.mp3", "*.flac", "*.m4a"):
    raw_audio_files.extend(glob.glob(os.path.join(RAW_AUDIO_DIR, ext)))

if raw_audio_files:
    logger.info(f"找到 {len(raw_audio_files)} 个原始音频文件:")
    for f in raw_audio_files[:5]: # 最多显示5个
        logger.info(f"  {f}")
    example_raw_audio_file = raw_audio_files[0] # 选择第一个作为示例
else:
    logger.warning(f"在 {RAW_AUDIO_DIR} 中没有找到音频文件。请添加一些音频文件以继续。")

### 2.1 加载并检查示例音频

In [ ]:
if example_raw_audio_file and os.path.exists(example_raw_audio_file):
    try:
        audio_data, sr = AudioUtils.load_audio(example_raw_audio_file) # 使用我们自己的加载函数
        duration = librosa.get_duration(y=audio_data, sr=sr)
        channels = audio_data.ndim
        
        logger.info(f"成功加载示例音频: {example_raw_audio_file}")
        logger.info(f"  采样率: {sr} Hz")
        logger.info(f"  时长: {duration:.2f} 秒")
        logger.info(f"  通道数: {channels}")
        logger.info(f"  数据类型: {audio_data.dtype}")
        logger.info(f"  数据范围: [{np.min(audio_data):.2f}, {np.max(audio_data):.2f}]")

        # 显示音频播放器
        print("原始音频播放：")
        ipd.display(ipd.Audio(data=audio_data, rate=sr))

        # 绘制波形图
        plt.figure(figsize=(12, 4))
        librosa.display.waveshow(audio_data, sr=sr)
        plt.title(f"原始音频波形: {os.path.basename(example_raw_audio_file)}")
        plt.xlabel("时间 (秒)")
        plt.ylabel("振幅")
        plt.tight_layout()
        plt.show()
        
    except Exception as e:
        logger.error(f"加载音频文件 {example_raw_audio_file} 失败: {e}")
else:
    logger.warning("没有有效的示例音频文件可供加载。")

## 3. 音频预处理：格式转换与重采样

ASR模型（如Whisper）通常期望特定格式的音频输入，例如：
- **采样率**: 16 kHz
- **通道**: 单声道 (Mono)
- **数据类型**: 32位浮点数 (float32)
- **格式**: WAV (通常是最兼容的)

我们将使用 `src.utils.audio_utils.AudioUtils.convert_to_target_format` 函数来执行这些转换，并将结果保存到 `data/processed_audio/` 目录。

In [ ]:
if example_raw_audio_file and os.path.exists(example_raw_audio_file):
    base_name = os.path.splitext(os.path.basename(example_raw_audio_file))[0]
    output_processed_filename = f"{base_name}_processed.wav"
    output_processed_path = os.path.join(PROCESSED_AUDIO_DIR, output_processed_filename)
    
    try:
        # 使用 AudioUtils 中的转换函数
        processed_audio_data, processed_sr = AudioUtils.convert_to_target_format(
            example_raw_audio_file,
            target_sr=TARGET_SAMPLE_RATE,
            target_channels=TARGET_CHANNELS,
            output_path=output_processed_path
        )
        logger.info(f"音频已处理并保存到: {output_processed_path}")
        
        # 检查处理后的音频属性
        duration_processed = librosa.get_duration(y=processed_audio_data, sr=processed_sr)
        channels_processed = processed_audio_data.ndim
        
        logger.info(f"处理后音频属性:")
        logger.info(f"  采样率: {processed_sr} Hz (目标: {TARGET_SAMPLE_RATE} Hz)")
        logger.info(f"  时长: {duration_processed:.2f} 秒")
        logger.info(f"  通道数: {channels_processed} (目标: {TARGET_CHANNELS})")
        logger.info(f"  数据类型: {processed_audio_data.dtype}")
        logger.info(f"  数据范围: [{np.min(processed_audio_data):.2f}, {np.max(processed_audio_data):.2f}]")
        
        print("处理后音频播放：")
        ipd.display(ipd.Audio(data=processed_audio_data, rate=processed_sr))
        
        plt.figure(figsize=(12, 4))
        librosa.display.waveshow(processed_audio_data, sr=processed_sr)
        plt.title(f"处理后音频波形: {output_processed_filename}")
        plt.xlabel("时间 (秒)")
        plt.ylabel("振幅")
        plt.tight_layout()
        plt.show()
        
    except Exception as e:
        logger.error(f"处理音频文件 {example_raw_audio_file} 失败: {e}")
else:
    logger.warning("没有有效的示例音频文件可供处理。")

## 4. （可选）演示音频分段

在流式 ASR 系统中，音频通常会被分割成小块进行处理。这里我们简要演示如何使用 `AudioSegmenter`。
这主要用于理解其功能，实际的流式处理在 `StreamingPipeline` 中进行。

In [ ]:
if 'processed_audio_data' in locals() and 'processed_sr' in locals():
    # 初始化分段器 (使用默认参数，或从 config.py 导入)
    # VAD 参数
    VAD_MIN_SILENCE_MS = 500 # 来自 config.py (示例值)
    VAD_SILENCE_THRESH_DBFS = -40 # 来自 config.py (示例值)
    MIN_SPEECH_DURATION_S = 0.3 # 来自 config.py (示例值)
    
    # 固定块参数
    CHUNK_SECONDS = 5.0 # 来自 config.py (示例值)
    OVERLAP_SECONDS = 1.0 # 来自 config.py (示例值)

    segmenter_vad = AudioSegmenter(
        use_vad=True, 
        min_silence_len_ms=VAD_MIN_SILENCE_MS, 
        silence_thresh_dbfs=VAD_SILENCE_THRESH_DBFS,
        min_speech_duration_s=MIN_SPEECH_DURATION_S
    )
    segmenter_fixed = AudioSegmenter(
        use_vad=False, 
        chunk_duration_s=CHUNK_SECONDS, 
        overlap_duration_s=OVERLAP_SECONDS
    )

    logger.info("演示基于 VAD 的分段：")
    # AudioSegmenter.segment_by_silence 需要 pydub.AudioSegment 对象
    # 我们需要从 numpy 数组转换回 pydub AudioSegment (虽然 AudioSegmenter 内部会处理 numpy)
    # 为了直接调用其核心方法，我们先转换
    # 注意：AudioSegmenter 类内部设计为处理 numpy 数组，但其 `segment_by_silence` 原始方法接受 AudioSegment
    # StreamingPipeline 中的用法是直接送 numpy 数组给 `process_audio`
    temp_pydub_audio = AudioSegment(
        processed_audio_data.tobytes(), 
        frame_rate=processed_sr, 
        sample_width=processed_audio_data.dtype.itemsize, 
        channels=1
    )
    vad_segments = segmenter_vad.segment_by_silence(temp_pydub_audio) # (start_ms, end_ms, audio_chunk_pydub)
    logger.info(f"  通过VAD找到 {len(vad_segments)} 个语音片段。")
    for i, (start_ms, end_ms, _) in enumerate(vad_segments[:3]): # 显示前3个
        logger.info(f"    片段 {i+1}: {start_ms/1000:.2f}s - {end_ms/1000:.2f}s")

    logger.info("
演示基于固定长度的分段：")
    # fixed_chunks 是 (numpy_chunk, start_time_s, end_time_s) 的列表
    # AudioSegmenter.segment_by_fixed_chunks 接受 numpy 数组和采样率
    fixed_chunks_info = list(segmenter_fixed.segment_by_fixed_chunks(processed_audio_data, processed_sr))
    logger.info(f"  通过固定块方法找到 {len(fixed_chunks_info)} 个音频块。")
    for i, (chunk_data, start_s, end_s) in enumerate(fixed_chunks_info[:3]): # 显示前3个
        logger.info(f"    块 {i+1}: {start_s:.2f}s - {end_s:.2f}s, 长度: {len(chunk_data)/processed_sr:.2f}s")
else:
    logger.warning("没有处理后的音频数据可用于演示分段。")

## 5. 文本记录处理

对于评估 ASR 性能，我们需要参考文本记录。这些文本文件应与处理后的音频文件对应，通常具有相同的文件名（扩展名不同，如 `.txt`），并存放在 `data/transcripts/` 目录下。

例如，如果处理后的音频是 `data/processed_audio/myaudio_processed.wav`，那么对应的文本记录应该是 `data/transcripts/myaudio_processed.txt`。

文本文件应为纯文本格式，UTF-8 编码。

In [ ]:
if 'output_processed_filename' in locals():
    base_processed_name = os.path.splitext(output_processed_filename)[0]
    example_transcript_filename = f"{base_processed_name}.txt"
    example_transcript_path = os.path.join(TRANSCRIPTS_DIR, example_transcript_filename)
    
    # 尝试创建一个虚拟的文本文件（如果不存在）
    if not os.path.exists(example_transcript_path):
        dummy_text = "This is a dummy transcript corresponding to the processed audio. Replace with actual text."
        with open(example_transcript_path, 'w', encoding='utf-8') as f:
            f.write(dummy_text)
        logger.info(f"创建了虚拟文本文件: {example_transcript_path}")
    else:
        logger.info(f"文本文件 {example_transcript_path} 已存在。")

    # 演示加载文本记录
    try:
        with open(example_transcript_path, 'r', encoding='utf-8') as f:
            transcript_content = f.read()
        logger.info(f"成功加载示例文本记录: {example_transcript_path}")
        logger.info(f"  内容: '{transcript_content[:100]}...'") # 显示前100个字符
    except FileNotFoundError:
        logger.warning(f"文本记录文件未找到: {example_transcript_path}")
    except Exception as e:
        logger.error(f"加载文本记录 {example_transcript_path} 失败: {e}")
else:
    logger.warning("无法确定示例处理文件名，跳过文本记录演示。")

## 6. 批量处理音频文件

以下是一个如何批量处理 `data/raw_audio/` 目录下所有音频文件的示例代码片段。
它将执行加载、转换、保存处理后音频的操作。您可以根据需要调整此代码。

In [ ]:
def batch_process_audios(raw_dir, processed_dir, target_sr, target_channels):
    logger.info(f"开始批量处理音频从 {raw_dir} 到 {processed_dir}")
    audio_files_to_process = []
    for ext in ("*.wav", "*.mp3", "*.flac", "*.m4a"): # 可以根据需要扩展支持的格式
        audio_files_to_process.extend(glob.glob(os.path.join(raw_dir, ext)))

    if not audio_files_to_process:
        logger.warning(f"在 {raw_dir} 中没有找到要处理的音频文件。")
        return

    processed_count = 0
    failed_count = 0
    for audio_file_path in audio_files_to_process:
        try:
            logger.info(f"正在处理: {audio_file_path}")
            base_name = os.path.splitext(os.path.basename(audio_file_path))[0]
            output_filename = f"{base_name}_processed.wav"
            output_path = os.path.join(processed_dir, output_filename)

            AudioUtils.convert_to_target_format(
                audio_file_path,
                target_sr=target_sr,
                target_channels=target_channels,
                output_path=output_path
            )
            logger.info(f"  已保存到: {output_path}")
            processed_count += 1
        except Exception as e:
            logger.error(f"  处理文件 {audio_file_path} 失败: {e}")
            failed_count += 1
            
    logger.info(f"批量处理完成。成功处理 {processed_count} 个文件，失败 {failed_count} 个文件。")

# 要执行批量处理，取消以下行的注释：
# batch_process_audios(RAW_AUDIO_DIR, PROCESSED_AUDIO_DIR, TARGET_SAMPLE_RATE, TARGET_CHANNELS)

logger.info("数据预处理 Notebook 执行完毕（批量处理部分已注释，请按需取消注释运行）。")

---